<a href="https://colab.research.google.com/github/zarduya/SQL-DB-PROJECT/blob/main/sql_database.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ayrus Pharmacy: Randomly Generated Retail Database
This notebook generates a realistic synthetic SQLite database for a fictional UK
health and beauty retailer, for my 7PAM2005 Data Mining & Discovery SQL assignment.
All data is randomly generated; no real customer data is used.

In [196]:
!pip install faker


In [197]:
import sqlite3
import random
import numpy as np
import pandas as pd
from faker import Faker

In [198]:
# set the seed so the random data comes out identical every time we run the notebook
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker('en_GB')   # for british names
Faker.seed(SEED)

In [199]:
from datetime import date

# some fixed variables ?
BRAND = "Ayrus Pharmacy"
OWN_BRAND = "Ayrus"
N_PRODUCTS = 250
DATA_START = date(2025, 1, 1)
DATA_END = date(2025, 12, 31)

# regions get their own table so that each name is typed only once
regions_df = pd.DataFrame({
    'region_id': [1, 2, 3, 4, 5],
    'region_name': ['London', 'South East', 'East of England',
                    'West Midlands', 'North West']
})
regions_df

,region_id,region_name
0,1,London
1,2,South East
2,3,East of England
3,4,West Midlands
4,5,North West


In [200]:
# stores are typed in manually since a real company already knows where its stores are
# format is ordinal: Local < High Street < Flagship
STORE_LIST = [
    ('Oxford Street', 'London', 1, 'Flagship'),
    ('Croydon', 'London', 1, 'High Street'),
    ('Stratford', 'London', 1, 'High Street'),
    ('Brighton', 'Brighton', 2, 'High Street'),
    ('Reading', 'Reading', 2, 'High Street'),
    ('Oxford', 'Oxford', 2, 'Local'),
    ('Luton', 'Luton', 3, 'High Street'),
    ('Cambridge', 'Cambridge', 3, 'High Street'),
    ('Watford', 'Watford', 3, 'Local'),
    ('Norwich', 'Norwich', 3, 'Local'),
    ('Birmingham Bullring', 'Birmingham', 4, 'Flagship'),
    ('Coventry', 'Coventry', 4, 'Local'),
    ('Manchester Arndale', 'Manchester', 5, 'Flagship'),
    ('Liverpool', 'Liverpool', 5, 'High Street'),
    ('Preston', 'Preston', 5, 'Local'),
]

# typical floor size for each format as (average, spread) in m²
# normal distribution so stores of the same format are similar but not identical
SIZE_BY_FORMAT = {'Local': (150, 30), 'High Street': (400, 80), 'Flagship': (1200, 200)}

stores = []
for store_id, (label, city, region_id, fmt) in enumerate(STORE_LIST, start=1):
    mean, sd = SIZE_BY_FORMAT[fmt]
    size = round(max(np.random.normal(mean, sd), 50), 1)   # never smaller than 50 m²
    opened = fake.date_between(start_date=date(1985, 1, 1), end_date=date(2022, 12, 31))
    stores.append({
        'store_id': store_id,
        'store_name': f"{BRAND} {label}",
        'city': city,
        'region_id': region_id,            # foreign key to regions
        'store_format': fmt,               # ordinal
        'size_sq_m': size,                 # ratio, 0 m² would mean no space at all
        'opened_date': opened.isoformat()  # interval, saved as 'YYYY-MM-DD' text since SQLite has no date type
    })

stores_df = pd.DataFrame(stores)
stores_df

,store_id,store_name,city,region_id,store_format,size_sq_m,opened_date
0,1,Ayrus Pharmacy Oxford Street,London,1,Flagship,1299.3,2009-04-18
1,2,Ayrus Pharmacy Croydon,London,1,High Street,388.9,1985-12-14
2,3,Ayrus Pharmacy Stratford,London,1,High Street,451.8,1995-06-14
3,4,Ayrus Pharmacy Brighton,Brighton,2,High Street,521.8,1993-06-25
4,5,Ayrus Pharmacy Reading,Reading,2,High Street,381.3,2012-12-25
5,6,Ayrus Pharmacy Oxford,Oxford,2,Local,143.0,2010-09-18
6,7,Ayrus Pharmacy Luton,Luton,3,High Street,526.3,2018-11-25
7,8,Ayrus Pharmacy Cambridge,Cambridge,3,High Street,461.4,1988-04-21
8,9,Ayrus Pharmacy Watford,Watford,3,Local,135.9,2001-01-12
9,10,Ayrus Pharmacy Norwich,Norwich,3,Local,166.3,1986-02-18


In [201]:
# each category has its own brands, items and sizes so the product names make sense
# real brand names are used to look real, but all prices and sales are made up
# share = how much of the 250 products are in this category
# median_price = normal price for the smallest size, in £
CATEGORIES = {
    'Skincare': {'brands': ['CeraVe', 'Nivea', 'La Roche-Posay', 'Garnier'],
                 'items': ['Moisturising Cream', 'Hydrating Cleanser', 'Vitamin C Serum', 'Night Cream', 'SPF 50 Sun Cream'],
                 'sizes': ['50ml', '100ml', '200ml'], 'median_price': 12.0, 'share': 0.20, 'own_brand': True},
    'Haircare': {'brands': ['Pantene', 'TRESemmé', 'Aussie', 'Herbal Essences'],
                 'items': ['Shampoo', 'Conditioner', 'Hair Mask', 'Dry Shampoo'],
                 'sizes': ['250ml', '400ml', '500ml'], 'median_price': 4.5, 'share': 0.14, 'own_brand': True},
    'Vitamins': {'brands': ['Seven Seas', 'Centrum', 'Vitabiotics'],
                 'items': ['Multivitamin Tablets', 'Vitamin D 25µg', 'Vitamin C 1000mg', 'Omega-3 Capsules', 'Iron Tablets'],
                 'sizes': ['30 pack', '60 pack', '90 pack'], 'median_price': 7.0, 'share': 0.14, 'own_brand': True},
    'Pain Relief': {'brands': ['Nurofen', 'Panadol', 'Anadin'],
                 'items': ['Tablets', 'Caplets', 'Extra Strength Tablets', 'Liquid Capsules'],
                 'sizes': ['8 pack', '16 pack', '32 pack'], 'median_price': 3.0, 'share': 0.10, 'own_brand': True},
    'Cold & Flu': {'brands': ['Lemsip', 'Beechams'],
                 'items': ['Max Strength Capsules', 'Hot Lemon Sachets', 'Day & Night Capsules', 'Cold & Flu Tablets'],
                 'sizes': ['5 pack', '10 pack', '16 pack'], 'median_price': 5.0, 'share': 0.10, 'own_brand': True},
    'Oral Care': {'brands': ['Colgate', 'Sensodyne', 'Aquafresh'],
                 'items': ['Whitening Toothpaste', 'Sensitive Toothpaste', 'Total Care Toothpaste', 'Fresh Mint Mouthwash', 'Alcohol-Free Mouthwash'],
                 'sizes': ['Single', 'Twin Pack'], 'median_price': 4.0, 'share': 0.08, 'own_brand': True},
    # no own brand perfume, pharmacies don't usually make their own perfume
    'Fragrance': {'brands': ['Hugo Boss', 'Calvin Klein', 'Paco Rabanne', 'Jimmy Choo'],
                 'items': ['Eau de Toilette', 'Eau de Parfum', 'Body Spray'],
                 'sizes': ['30ml', '50ml', '100ml'], 'median_price': 45.0, 'share': 0.12, 'own_brand': False},
    'Baby':     {'brands': ['Pampers', 'Huggies'],
                 'items': ['Nappies Size 2', 'Nappies Size 3', 'Nappies Size 4', 'Nappies Size 5', 'Sensitive Baby Wipes'],
                 'sizes': ['Single Pack', 'Value Pack', 'Jumbo Pack'], 'median_price': 9.0, 'share': 0.12, 'own_brand': True},
}

In [202]:
products = []
seen_names = set()    # remembers names already made so we don't get the same product twice
base_prices = {}      # one price per product, so all its sizes are priced in line
product_id = 1

# bigger sizes cost more, but cheaper per unit, like in real shops
SIZE_MULTIPLIERS = [1.0, 1.6, 2.2]

for category, info in CATEGORIES.items():
    n_in_category = round(N_PRODUCTS * info['share'])
    brands = info['brands'] + ([OWN_BRAND] if info['own_brand'] else [])
    created = 0

    while created < n_in_category:
        brand = random.choice(brands)
        item = random.choice(info['items'])
        size = random.choice(info['sizes'])
        name = f"{brand} {item} {size}"
        if name in seen_names:
            continue              # already made, try again
        seen_names.add(name)

        # pick the price once per product
        # log-normal means most things are cheap and only a few are expensive
        key = (brand, item)
        if key not in base_prices:
            base = np.random.lognormal(mean=np.log(info['median_price']), sigma=0.3)
            if brand == OWN_BRAND:
                base *= 0.7                      # own brand is about 30% cheaper
            base_prices[key] = base

        price = base_prices[key] * SIZE_MULTIPLIERS[info['sizes'].index(size)]
        price = max(round(price) - 0.01, 0.99)   # make prices look like £4.99

        products.append({
            'product_id': product_id,
            'product_name': name,
            'category': category,          # nominal
            'brand': brand,                # nominal
            'unit_price': price,           # ratio
            'is_own_brand': int(brand == OWN_BRAND)   # 1 = own brand, 0 = not
        })
        product_id += 1
        created += 1

products_df = pd.DataFrame(products)
print(len(products_df), "products generated")
products_df.sort_values('product_name').head(15)

250 products generated


,product_id,product_name,category,brand,unit_price,is_own_brand
139,140,Anadin Extra Strength Tablets 8 pack,Pain Relief,Anadin,2.99,0
126,127,Anadin Liquid Capsules 16 pack,Pain Relief,Anadin,3.99,0
144,145,Anadin Liquid Capsules 32 pack,Pain Relief,Anadin,5.99,0
132,133,Anadin Tablets 16 pack,Pain Relief,Anadin,3.99,0
141,142,Anadin Tablets 32 pack,Pain Relief,Anadin,5.99,0
185,186,Aquafresh Alcohol-Free Mouthwash Single,Oral Care,Aquafresh,3.99,0
188,189,Aquafresh Fresh Mint Mouthwash Single,Oral Care,Aquafresh,3.99,0
182,183,Aquafresh Fresh Mint Mouthwash Twin Pack,Oral Care,Aquafresh,6.99,0
187,188,Aquafresh Sensitive Toothpaste Twin Pack,Oral Care,Aquafresh,5.99,0
189,190,Aquafresh Total Care Toothpaste Single,Oral Care,Aquafresh,3.99,0


In [203]:
# check the products look right
print(products_df['category'].value_counts(), "\n")
print(products_df.groupby('category')['unit_price'].median().round(2), "\n")
print(products_df['unit_price'].describe().round(2))

category
Skincare       50
Haircare       35
Vitamins       35
Baby           30
Fragrance      30
Pain Relief    25
Cold & Flu     25
Oral Care      20
Name: count, dtype: int64 

category
Baby           11.99
Cold & Flu      6.99
Fragrance      69.49
Haircare        5.99
Oral Care       5.49
Pain Relief     3.99
Skincare       15.99
Vitamins       10.99
Name: unit_price, dtype: float64 

count    250.00
mean      17.98
std       24.54
min        1.99
25%        5.99
50%        9.99
75%       16.99
max      138.99
Name: unit_price, dtype: float64


# customers

In [204]:
N_CUSTOMERS = 1175   # real customers, about 25 copies get added later to make around 1,200

# more women than men shop at pharmacies, so the weights show that
GENDERS = ['Female', 'Male', 'Non-binary', 'Prefer not to say']
GENDER_WEIGHTS = [0.62, 0.34, 0.01, 0.03]

# most people are Bronze, only a few reach Platinum
TIERS = ['Bronze', 'Silver', 'Gold', 'Platinum']
TIER_WEIGHTS = [0.55, 0.28, 0.13, 0.04]

EMAIL_DOMAINS = ['gmail.com', 'outlook.com', 'hotmail.co.uk', 'yahoo.co.uk', 'icloud.com']
DOMAIN_WEIGHTS = [0.45, 0.20, 0.15, 0.10, 0.10]

# bigger stores get more customers, but square root stops the big ones getting too many
store_weights = np.sqrt(stores_df['size_sq_m'])
store_weights = store_weights / store_weights.sum()

def make_email(first, last, used_emails):
    """make an email from the person's name so name and email match"""
    f = ''.join(c for c in first.lower() if c.isalpha())   # remove things like ' and -
    l = ''.join(c for c in last.lower() if c.isalpha())
    pattern = random.choice(['{f}.{l}', '{f}{l}{n}', '{fi}{l}', '{f}_{l}{n}'])
    local = pattern.format(f=f, l=l, fi=f[0], n=random.randint(1, 99))
    domain = random.choices(EMAIL_DOMAINS, DOMAIN_WEIGHTS)[0]
    email = f"{local}@{domain}"
    while email in used_emails:            # if two people have the same name, add a number
        email = f"{local}{random.randint(100, 999)}@{domain}"
    used_emails.add(email)
    return email

In [205]:
customers = []
used_emails = set()   # remembers emails already used so none repeat

for customer_id in range(1, N_CUSTOMERS + 1):
    gender = random.choices(GENDERS, GENDER_WEIGHTS)[0]
    # first name matches gender so it looks real
    if gender == 'Female':
        first = fake.first_name_female()
    elif gender == 'Male':
        first = fake.first_name_male()
    else:
        first = fake.first_name()
    last = fake.last_name()

    # most people are around 43, kept between 16 and 90
    age = int(np.clip(np.random.normal(43, 16), 16, 90))

    # picking the customer's home store, bigger stores get picked more
    store_id = int(np.random.choice(stores_df['store_id'], p=store_weights))
    # customer lives in the same city as their store
    city = stores_df.loc[stores_df['store_id'] == store_id, 'city'].iloc[0]

    customers.append({
        'customer_id': customer_id,
        'first_name': first,                                   # nominal
        'last_name': last,                                     # nominal
        'email': make_email(first, last, used_emails),         # nominal, every email is different
        'phone': f"07{random.randint(100000000, 999999999)}",  # UK mobile style number
        'age': age,                                            # ratio
        'gender': gender,                                      # nominal
        'city': city,                                          # nominal
        'preferred_store_id': store_id,                        # foreign key to stores
        'loyalty_tier': random.choices(TIERS, TIER_WEIGHTS)[0],  # ordinal
        'signup_date': fake.date_between(date(2016, 1, 1), date(2025, 10, 31)).isoformat()  # interval
    })

customers_df = pd.DataFrame(customers)
print(len(customers_df), "customers generated")
customers_df.sample(10)

1175 customers generated


,customer_id,first_name,last_name,email,phone,age,gender,city,preferred_store_id,loyalty_tier,signup_date
570,571,Harriet,Freeman,harrietfreeman93@hotmail.co.uk,07586909856,62,Female,Manchester,13,Silver,2024-10-05
803,804,Sally,Brown,sally_brown56@outlook.com,07467435349,48,Female,Birmingham,11,Bronze,2018-01-13
695,696,Denise,Marshall,denise.marshall@icloud.com,07459048149,40,Female,Luton,7,Bronze,2022-04-12
409,410,Carly,O'Sullivan,carly.osullivan@gmail.com,07254460628,53,Female,Cambridge,8,Silver,2017-07-09
272,273,Suzanne,Bradley,suzannebradley63@outlook.com,07844567935,30,Female,Watford,9,Gold,2021-05-29
1171,1172,Jacqueline,Smith,jacqueline.smith@gmail.com,07537309736,39,Female,Brighton,4,Silver,2020-06-27
1157,1158,Karen,Byrne,karen.byrne@yahoo.co.uk,07331478831,32,Female,London,3,Bronze,2024-08-22
343,344,Shannon,Hill,shannon.hill@hotmail.co.uk,07240346956,22,Female,London,1,Bronze,2024-12-24
232,233,Valerie,Thomas,valerie_thomas11@hotmail.co.uk,07417161302,52,Female,Manchester,13,Silver,2016-07-14
1112,1113,Jordan,Haynes,jhaynes832@yahoo.co.uk,07805403426,81,Male,London,1,Bronze,2025-09-08


In [206]:
# check the customers look right
print(customers_df['gender'].value_counts(), "\n")
print(customers_df['loyalty_tier'].value_counts(), "\n")
print(customers_df['age'].describe().round(1), "\n")
print(customers_df['city'].value_counts(), "\n")
print("All emails unique:", customers_df['email'].is_unique)

gender
Female               695
Male                 411
Prefer not to say     56
Non-binary            13
Name: count, dtype: int64 

loyalty_tier
Bronze      652
Silver      328
Gold        145
Platinum     50
Name: count, dtype: int64 

count    1175.0
mean       43.5
std        15.1
min        16.0
25%        33.0
50%        43.0
75%        53.0
max        90.0
Name: age, dtype: float64 

city
London        308
Manchester    143
Birmingham    121
Luton          93
Brighton       85
Reading        72
Cambridge      71
Liverpool      61
Norwich        51
Coventry       49
Watford        48
Oxford         40
Preston        33
Name: count, dtype: int64 

All emails unique: True


In [207]:
# how many times each tier customers shops in 2025 on average, higher tiers shop more
VISITS_BY_TIER = {'Bronze': 2, 'Silver': 4, 'Gold': 7, 'Platinum': 12}
GUEST_SHARE = 0.15    # about 15% of sales are guests with no loyalty card

PAYMENT_METHODS = ['Contactless', 'Card', 'Mobile Pay', 'Cash', 'Gift Card']
PAYMENT_WEIGHTS = [0.50, 0.20, 0.18, 0.10, 0.02]
PAYMENT_WEIGHTS_OVER_65 = [0.40, 0.25, 0.05, 0.28, 0.02]   # older people use more cash

# most ratings are good, like real reviews
# every sale gets a rating here, about 70% get removed later in the missing data step
RATINGS = [1, 2, 3, 4, 5]
RATING_WEIGHTS = [0.04, 0.06, 0.15, 0.35, 0.40]

# shop open 8am to 8pm, busiest at lunch and after work
HOURS = list(range(8, 20))
HOUR_WEIGHTS = [3, 5, 6, 7, 10, 10, 8, 7, 7, 9, 8, 5]

# busier in December and January (Christmas and cold season), Saturday is the busiest day
all_days = pd.date_range(DATA_START, DATA_END, freq='D')
MONTH_FACTOR = {1: 1.10, 2: 1.00, 3: 0.95, 4: 0.95, 5: 1.00, 6: 1.05,
                7: 1.10, 8: 1.05, 9: 0.95, 10: 1.00, 11: 1.15, 12: 1.45}
WEEKDAY_FACTOR = {0: 0.90, 1: 0.90, 2: 0.95, 3: 1.00, 4: 1.15, 5: 1.40, 6: 1.10}  # 0 = Monday
day_weights = np.array([MONTH_FACTOR[d.month] * WEEKDAY_FACTOR[d.weekday()] for d in all_days])

def random_datetime(earliest=DATA_START):
    """pick a realistic date and time, never before 'earliest'
    so nobody buys something before they signed up"""
    valid = np.flatnonzero(all_days >= pd.Timestamp(earliest))
    w = day_weights[valid]
    day = all_days[np.random.choice(valid, p=w / w.sum())]
    hour = random.choices(HOURS, HOUR_WEIGHTS)[0]
    moment = day + pd.Timedelta(hours=hour, minutes=random.randint(0, 59), seconds=random.randint(0, 59))
    return moment.strftime('%Y-%m-%d %H:%M:%S')

In [208]:
transactions = []
store_ids = stores_df['store_id'].tolist()

# ---- sales from loyalty card members ----
for c in customers_df.itertuples():
    # how many times this person shops, based on their tier
    n_visits = np.random.poisson(VISITS_BY_TIER[c.loyalty_tier])
    # can't shop before they signed up or before 2025 starts
    earliest = max(date.fromisoformat(c.signup_date), DATA_START)
    pay_weights = PAYMENT_WEIGHTS_OVER_65 if c.age >= 65 else PAYMENT_WEIGHTS

    for _ in range(n_visits):
        # 85% of the time they shop at their own store, 15% somewhere else
        if random.random() < 0.85:
            store_id = int(c.preferred_store_id)
        else:
            store_id = int(np.random.choice(store_ids, p=store_weights))

        transactions.append({
            'customer_id': int(c.customer_id),                                  # foreign key to customers
            'store_id': store_id,                                               # foreign key to stores
            'transaction_datetime': random_datetime(earliest),                  # interval
            'payment_method': random.choices(PAYMENT_METHODS, pay_weights)[0],  # nominal
            'satisfaction_rating': random.choices(RATINGS, RATING_WEIGHTS)[0]   # ordinal
        })

# ---- guest sales, no customer linked ----
n_guest = round(len(transactions) * GUEST_SHARE / (1 - GUEST_SHARE))
for _ in range(n_guest):
    transactions.append({
        'customer_id': None,    # guest, so no customer
        'store_id': int(np.random.choice(store_ids, p=store_weights)),
        'transaction_datetime': random_datetime(),
        'payment_method': random.choices(PAYMENT_METHODS, PAYMENT_WEIGHTS)[0],
        'satisfaction_rating': random.choices(RATINGS, RATING_WEIGHTS)[0]
    })

# sort by time and number them in order, like a real till system
transactions_df = (pd.DataFrame(transactions)
                   .sort_values('transaction_datetime')
                   .reset_index(drop=True))
transactions_df.insert(0, 'transaction_id', range(1, len(transactions_df) + 1))

# keep customer_id as whole numbers with proper blanks, not 1042.0
transactions_df['customer_id'] = transactions_df['customer_id'].astype('Int64')

print(len(transactions_df), "transactions generated")
transactions_df.sample(10)

4945 transactions generated


,transaction_id,customer_id,store_id,transaction_datetime,payment_method,satisfaction_rating
1593,1594,121,1,2025-05-12 18:43:02,Card,5
504,505,<NA>,11,2025-02-10 17:48:51,Contactless,3
3706,3707,498,13,2025-10-19 15:52:36,Mobile Pay,4
2281,2282,559,14,2025-07-04 19:08:27,Cash,5
3424,3425,641,15,2025-09-28 14:50:53,Contactless,2
1841,1842,836,12,2025-06-02 17:46:54,Contactless,4
3887,3888,738,7,2025-11-03 17:52:27,Contactless,5
796,797,557,13,2025-03-07 09:39:00,Contactless,4
3379,3380,764,9,2025-09-23 15:52:02,Contactless,5
2579,2580,15,4,2025-07-26 11:39:13,Contactless,5


In [209]:
# check the transactions look right
print("Guest share:", round(transactions_df['customer_id'].isna().mean(), 3), "\n")

dt = pd.to_datetime(transactions_df['transaction_datetime'])
print("Sales by month:\n", dt.dt.month.value_counts().sort_index(), "\n")
print("Sales by weekday (0 = Monday):\n", dt.dt.weekday.value_counts().sort_index(), "\n")
print("Sales by hour:\n", dt.dt.hour.value_counts().sort_index(), "\n")
print(transactions_df['payment_method'].value_counts(), "\n")
print(transactions_df['store_id'].value_counts().sort_index())

Guest share: 0.15 

Sales by month:
 transaction_datetime
1     373
2     338
3     386
4     341
5     377
6     414
7     431
8     423
9     366
10    397
11    498
12    601
Name: count, dtype: int64 

Sales by weekday (0 = Monday):
 transaction_datetime
0    636
1    619
2    644
3    637
4    737
5    897
6    775
Name: count, dtype: int64 

Sales by hour:
 transaction_datetime
8     183
9     315
10    347
11    449
12    530
13    582
14    464
15    407
16    412
17    522
18    457
19    277
Name: count, dtype: int64 

payment_method
Contactless    2445
Card           1017
Mobile Pay      835
Cash            559
Gift Card        89
Name: count, dtype: int64 

store_id
1     655
2     276
3     306
4     400
5     268
6     183
7     388
8     350
9     208
10    172
11    498
12    208
13    606
14    280
15    147
Name: count, dtype: int64


In [210]:
# quick lookups for each product, used to find related products
item_of = {}
for p in products_df.itertuples():
    items = CATEGORIES[p.category]['items']
    # picking the longest match so "Dry Shampoo" doesn't count as "Shampoo"
    item_of[p.product_id] = max([i for i in items if i in p.product_name], key=len)
category_of = dict(zip(products_df['product_id'], products_df['category']))
brand_of = dict(zip(products_df['product_id'], products_df['brand']))
price_of = dict(zip(products_df['product_id'], products_df['unit_price']))
product_ids = products_df['product_id'].tolist()

# how popular each category is in general
CATEGORY_POPULARITY = {'Skincare': 1.2, 'Haircare': 1.0, 'Vitamins': 0.8, 'Pain Relief': 1.0,
                       'Cold & Flu': 0.7, 'Oral Care': 0.9, 'Fragrance': 0.4, 'Baby': 0.6}

# some products are best sellers and some barely sell, like a real shop
product_popularity = np.random.lognormal(0, 0.6, len(product_ids))

def seasonal_factor(pid, month):
    """some things sell more at certain times of year"""
    cat = category_of[pid]
    if cat == 'Cold & Flu':      # winter up, summer down
        return 2.5 if month in (1, 2, 11, 12) else (0.5 if month in (6, 7, 8) else 1.0)
    if cat == 'Fragrance':       # Christmas gifts
        return 3.0 if month == 12 else (1.5 if month == 11 else 1.0)
    if item_of[pid] == 'SPF 50 Sun Cream':   # summer
        return 3.0 if month in (5, 6, 7, 8) else 0.4
    return 1.0

# work out the chance of each product being picked in each month, done once so it runs fast
month_weights = {}
for m in range(1, 13):
    w = np.array([CATEGORY_POPULARITY[category_of[pid]] * product_popularity[i] * seasonal_factor(pid, m)
                  for i, pid in enumerate(product_ids)])
    month_weights[m] = w / w.sum()

# things people often buy together: (if basket has this, then add this, chance)
# these patterns can be found later with association rules
COMPANION_RULES = [
    ('Shampoo', 'Conditioner', 0.45),
    ('Nappies', 'Sensitive Baby Wipes', 0.50),
    ('Hydrating Cleanser', 'Moisturising Cream', 0.35),
    ('Cold & Flu', 'Vitamin C 1000mg', 0.30),
    ('Cold & Flu', 'Pain Relief', 0.20),
]

def matches(pid, key):
    """true if the product's item starts with key, or its category is key"""
    return item_of[pid].startswith(key) or category_of[pid] == key

def pick_companion(trigger_pid, key):
    """pick a related product, usually the same brand if there is one"""
    options = [pid for pid in product_ids if matches(pid, key)]
    same_brand = [pid for pid in options if brand_of[pid] == brand_of[trigger_pid]]
    if same_brand and random.random() < 0.6:
        return random.choice(same_brand)
    return random.choice(options)

# most people buy just 1 of each thing
QUANTITIES = [1, 2, 3, 4]
QUANTITY_WEIGHTS = [0.80, 0.14, 0.04, 0.02]

# about 12% of items are on offer
PROMO_CHANCE = 0.12
PROMO_DISCOUNTS = [0.10, 0.25, 0.33]   # 10% off, 25% off, 3 for 2

In [211]:
items = []

for t in transactions_df.itertuples():
    month = int(t.transaction_datetime[5:7])   # month number from the date, e.g. '2025-12-..' gives 12

    # how many products to pick at random, most baskets get 1 to 3
    n_picks = 1 + np.random.poisson(1.2)
    # pick products using that month's chances, a set means no product appears twice
    basket = set(int(x) for x in np.random.choice(product_ids, size=n_picks, p=month_weights[month]))

    # add related products using the companion rules
    for pid in list(basket):
        for trigger, companion, chance in COMPANION_RULES:
            if matches(pid, trigger) and random.random() < chance:
                basket.add(pick_companion(pid, companion))

    # one row for each product in the basket
    for pid in basket:
        price = price_of[pid]
        if random.random() < PROMO_CHANCE:
            price = round(price * (1 - random.choice(PROMO_DISCOUNTS)), 2)   # item on offer
        items.append({
            'transaction_id': int(t.transaction_id),   # foreign key to transactions, part of composite key
            'product_id': pid,                         # foreign key to products, part of composite key
            'quantity': random.choices(QUANTITIES, QUANTITY_WEIGHTS)[0],   # ratio
            'unit_price_at_sale': price                # ratio
        })

transaction_items_df = pd.DataFrame(items)
print(len(transaction_items_df), "items generated")
print(transaction_items_df.head(10))

12278 items generated
   transaction_id  product_id  quantity  unit_price_at_sale
0               1          24         1               18.99
1               1         161         1                8.99
2               1           3         1               28.99
3               1          23         1               13.99
4               2         105         1                6.99
5               3          99         1               13.99
6               3         155         3                7.99
7               3         159         1                6.99
8               4         193         1               89.99
9               4          87         1                8.99


In [212]:
# check the items look right
print("Average items per basket:", round(transaction_items_df.groupby('transaction_id').size().mean(), 2))
print("Every transaction has items:", transaction_items_df['transaction_id'].nunique() == len(transactions_df))
print("Composite key unique:", not transaction_items_df.duplicated(['transaction_id', 'product_id']).any(), "\n")
print(transaction_items_df['quantity'].value_counts(), "\n")

# join items with products and dates to check the seasons worked
merged = (transaction_items_df
          .merge(products_df[['product_id', 'category', 'product_name']], on='product_id')
          .merge(transactions_df[['transaction_id', 'transaction_datetime']], on='transaction_id'))
merged['month'] = merged['transaction_datetime'].str[5:7].astype(int)
print(pd.crosstab(merged['month'], merged['category']), "\n")

# check shampoo and conditioner are often bought together
shampoo_baskets = set(merged.loc[merged['product_name'].str.contains(' Shampoo ') &
                                 ~merged['product_name'].str.contains('Dry'), 'transaction_id'])
conditioner_baskets = set(merged.loc[merged['product_name'].str.contains('Conditioner'), 'transaction_id'])
print("Baskets with shampoo:", len(shampoo_baskets))
print("...of those, also had conditioner:", round(len(shampoo_baskets & conditioner_baskets) / len(shampoo_baskets), 2))

Average items per basket: 2.48
Every transaction has items: True
Composite key unique: True 

quantity
1    9764
2    1755
3     503
4     256
Name: count, dtype: int64 

category  Baby  Cold & Flu  Fragrance  Haircare  Oral Care  Pain Relief  \
month                                                                     
1          115         137         63       156         45          110   
2          114         117         55       133         43          103   
3          126          79         66       195         65          140   
4           88          53         49       156         48          120   
5           79          54         47       140         57           94   
6           80          26         45       159         65          107   
7           92          35         62       161         61          123   
8          118          30         64       175         65          117   
9           85          58         46       164         73          131   
10  

## Store Inventory

In [213]:
from datetime import timedelta

# small shops don't stock everything, flagships stock the full range
RANGE_BY_FORMAT = {'Local': 0.60, 'High Street': 0.80, 'Flagship': 1.00}
# bigger stores keep more of each product on the shelf
STOCK_SCALE = {'Local': 6, 'High Street': 12, 'Flagship': 25}
SNAPSHOT_DATE = DATA_END   # stock is counted on the last day of 2025

popularity_of = dict(zip(product_ids, product_popularity))   # reuse best seller scores from before

inventory = []
for s in stores_df.itertuples():
    for pid in product_ids:
        # skip products this store doesn't sell
        if random.random() > RANGE_BY_FORMAT[s.store_format]:
            continue

        # popular products and bigger stores have more stock
        avg_stock = STOCK_SCALE[s.store_format] * popularity_of[pid]
        stock = int(np.random.poisson(avg_stock))
        if random.random() < 0.03:
            stock = 0              # a few items have sold out

        # most items restocked in the last 3 weeks, sold out ones have waited longer
        days_ago = random.randint(15, 60) if stock == 0 else random.randint(0, 21)

        inventory.append({
            'store_id': int(s.store_id),    # foreign key to stores, part of composite key
            'product_id': pid,              # foreign key to products, part of composite key
            'stock_level': stock,           # ratio, 0 means none left
            'last_restock_date': (SNAPSHOT_DATE - timedelta(days=days_ago)).isoformat()  # interval
        })

store_inventory_df = pd.DataFrame(inventory)

# check the inventory looks right
print(len(store_inventory_df), "inventory rows generated")
print("Composite key unique:", not store_inventory_df.duplicated(['store_id', 'product_id']).any())
print("Out of stock share:", round((store_inventory_df['stock_level'] == 0).mean(), 3), "\n")
print("Products stocked per store:\n", store_inventory_df['store_id'].value_counts().sort_index(), "\n")
print(store_inventory_df['stock_level'].describe().round(1))

2938 inventory rows generated
Composite key unique: True
Out of stock share: 0.039 

Products stocked per store:
 store_id
1     250
2     197
3     198
4     198
5     202
6     159
7     207
8     208
9     150
10    164
11    250
12    154
13    250
14    194
15    157
Name: count, dtype: int64 

count    2938.0
mean       16.5
std        15.6
min         0.0
25%         6.0
50%        12.0
75%        22.0
max       133.0
Name: stock_level, dtype: float64


In [214]:
import math

# London is a bit warmer (city heat), the North West a bit cooler
REGION_OFFSET = {1: 1.0, 2: 0.5, 3: 0.0, 4: -0.5, 5: -1.0}

weather = []
for r in regions_df.itertuples():
    noise = 0.0
    for d in all_days:
        # normal temperature for the time of year, about 4°C in January and 17°C in July
        seasonal = 11 + 6.5 * math.sin(2 * math.pi * (d.dayofyear - 110) / 365)
        # today's weather is a lot like yesterday's, so part of the noise carries over
        noise = 0.7 * noise + np.random.normal(0, 1.5)
        temp = round(float(np.clip(seasonal + REGION_OFFSET[r.region_id] + noise, -15, 40)), 1)

        weather.append({
            'region_id': int(r.region_id),             # foreign key to regions, part of composite key
            'weather_date': d.strftime('%Y-%m-%d'),    # interval, part of composite key
            'avg_temp_c': temp                         # interval, 0°C doesn't mean "no temperature"
        })

daily_weather_df = pd.DataFrame(weather)

# check the weather looks right
print(len(daily_weather_df), "weather rows generated")
print("Composite key unique:", not daily_weather_df.duplicated(['region_id', 'weather_date']).any(), "\n")
daily_weather_df['month'] = daily_weather_df['weather_date'].str[5:7].astype(int)
print(daily_weather_df.pivot_table(index='month', columns='region_id', values='avg_temp_c').round(1))
daily_weather_df = daily_weather_df.drop(columns='month')   # only needed for the check

1825 weather rows generated
Composite key unique: True 

region_id     1     2     3     4     5
month                                  
1           6.5   4.7   5.8   4.0   4.5
2           7.6   6.9   5.5   5.9   4.6
3           7.3   7.4   7.2   7.2   6.4
4          12.5  12.5  10.2   9.9   8.3
5          15.0  14.1  14.2  13.2  13.5
6          16.5  16.6  17.2  16.4  15.1
7          19.4  17.8  18.8  15.3  17.5
8          17.9  18.1  16.7  16.2  13.8
9          13.0  14.4  13.6  14.9  14.5
10         12.6  12.8  12.3  11.1   8.7
11          9.4   8.8  10.2   8.0   7.1
12          5.6   4.7   4.3   5.6   4.3


##Deliberate Missing Data

In [215]:
# ---- DELIBERATE MISSING DATA ----
# real data always has gaps, so some values are removed on purpose
# each line shows how much is removed and why

# turn these columns into types that can hold blanks properly (whole numbers stay whole)
customers_df['age'] = customers_df['age'].astype('Int64')
transactions_df['satisfaction_rating'] = transactions_df['satisfaction_rating'].astype('Int64')
daily_weather_df['avg_temp_c'] = daily_weather_df['avg_temp_c'].astype('Float64')

def add_missing(df, column, fraction, reason):
    """blank out a random part of a column and print what was done"""
    rows = df.sample(frac=fraction).index  # uses the notebook seed, so still the same every run
    df.loc[rows, column] = pd.NA
    print(f"{column}: {len(rows)} values removed ({fraction:.0%}), {reason}")

add_missing(customers_df, 'phone', 0.06, 'people skipped the optional phone field')
add_missing(customers_df, 'age', 0.04, 'people chose not to give their age')
add_missing(transactions_df, 'satisfaction_rating', 0.70, 'most customers never leave a rating')
add_missing(daily_weather_df, 'avg_temp_c', 0.01, 'weather station was offline')

# guest sales already have no customer_id, that blank is by design, not added here
print("\ncustomer_id blanks from guest sales:", transactions_df['customer_id'].isna().sum())

phone: 70 values removed (6%), people skipped the optional phone field
age: 47 values removed (4%), people chose not to give their age
satisfaction_rating: 3462 values removed (70%), most customers never leave a rating
avg_temp_c: 18 values removed (1%), weather station was offline

customer_id blanks from guest sales: 742


In [216]:
# ---- DELIBERATE DUPLICATES ----
# real databases have duplicates, so some are added on purpose
# they get new IDs, so the database rules don't block them, just like in real life

N_DUP_CUSTOMERS = 25       # about 2% of customers signed up twice
N_DUP_TRANSACTIONS = 25    # about 0.5% of sales recorded twice by a till glitch

def typo(name):
    """swap two letters next to each other, like a typing mistake"""
    if len(name) < 4:
        return name + name[-1]
    i = random.randint(1, len(name) - 2)
    return name[:i] + name[i + 1] + name[i] + name[i + 2:]

# ---- duplicate customers ----
dup_customers = []
customer_pairs = []    # (original id, duplicate id) so we can show them later
next_id = int(customers_df['customer_id'].max()) + 1

for row in customers_df.sample(n=N_DUP_CUSTOMERS).itertuples(index=False):
    dup = row._asdict()
    local, domain = dup['email'].split('@')
    change = random.choice(['email_case', 'new_email', 'name_typo'])

    if change == 'email_case':      # same email typed with a capital letter
        dup['email'] = local.capitalize() + '@' + domain
    elif change == 'new_email':     # signed up again with a different email provider
        other = random.choice([d for d in EMAIL_DOMAINS if d != domain])
        dup['email'] = local + '@' + other
    else:                           # name typed wrong at the till
        dup['first_name'] = typo(dup['first_name'])
        dup['email'] = local.capitalize() + '@' + domain

    while dup['email'] in used_emails:   # just in case the new email already exists
        dup['email'] = f"{local}{random.randint(100, 999)}@{domain}"
    used_emails.add(dup['email'])

    dup['customer_id'] = next_id
    dup['loyalty_tier'] = 'Bronze'       # a new account starts at the bottom tier
    original_signup = date.fromisoformat(row.signup_date)
    start = min(original_signup + timedelta(days=30), DATA_END)
    dup['signup_date'] = fake.date_between(start, DATA_END).isoformat()   # signed up again later

    dup_customers.append(dup)
    customer_pairs.append((row.customer_id, next_id))
    next_id += 1

customers_df = pd.concat([customers_df, pd.DataFrame(dup_customers).astype(customers_df.dtypes.to_dict())], ignore_index=True)
customers_df['age'] = customers_df['age'].astype('Int64')   # keep ages as whole numbers

# ---- duplicate transactions (till glitch) ----
new_tx, new_items = [], []
tx_pairs = []
next_tid = int(transactions_df['transaction_id'].max()) + 1

for row in transactions_df.sample(n=N_DUP_TRANSACTIONS).itertuples(index=False):
    dup = row._asdict()
    dup['transaction_id'] = next_tid      # same sale, new ID
    new_tx.append(dup)

    # copy the basket too, so the duplicate sale has the same items
    basket = transaction_items_df[transaction_items_df['transaction_id'] == row.transaction_id].copy()
    basket['transaction_id'] = next_tid
    new_items.append(basket)

    tx_pairs.append((row.transaction_id, next_tid))
    next_tid += 1

transactions_df = pd.concat([transactions_df, pd.DataFrame(new_tx).astype(transactions_df.dtypes.to_dict())], ignore_index=True)
transaction_items_df = pd.concat([transaction_items_df] + new_items, ignore_index=True)
transactions_df['customer_id'] = transactions_df['customer_id'].astype('Int64')
transactions_df['satisfaction_rating'] = transactions_df['satisfaction_rating'].astype('Int64')

# ---- show what was added ----
print(f"Added {len(dup_customers)} duplicate customers and {len(new_tx)} duplicate transactions\n")
print("Example duplicate customers (original first, then duplicate):")
for orig, dup_id in customer_pairs[:3]:
    print(customers_df.loc[customers_df['customer_id'].isin([orig, dup_id]),
                           ['customer_id', 'first_name', 'last_name', 'email', 'phone', 'signup_date']], "\n")
print("Example duplicate transactions:")
print(transactions_df[transactions_df['transaction_id'].isin(tx_pairs[0])], "\n")
print("Totals now:", len(customers_df), "customers,", len(transactions_df), "transactions,",
      len(transaction_items_df), "items")
print("Emails still unique:", customers_df['email'].is_unique)

Added 25 duplicate customers and 25 duplicate transactions

Example duplicate customers (original first, then duplicate):
      customer_id first_name last_name                       email  \
407           408    Phillip    Taylor  phillip.taylor@outlook.com   
1175         1176    Phillip    Taylor  Phillip.taylor@outlook.com   

            phone signup_date  
407   07208437656  2023-02-02  
1175  07208437656  2024-07-20   

      customer_id first_name last_name                      email  \
308           309       Gail   Francis  gail_francis45@icloud.com   
1176         1177       Gali   Francis  Gail_francis45@icloud.com   

            phone signup_date  
308   07437695739  2019-03-25  
1176  07437695739  2022-04-17   

      customer_id first_name last_name                    email        phone  \
664           665      Billy     Smith  billy_smith94@gmail.com  07840076530   
1177         1178      Bilyl     Smith  Billy_smith94@gmail.com  07840076530   

     signup_date  
664

In [217]:
import os

DB_FILE = 'ayrus_pharmacy.db'

# delete any old copy first so every run starts from a clean database
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)

conn = sqlite3.connect(DB_FILE)
conn.execute('PRAGMA foreign_keys = ON')   # SQLite ignores foreign keys unless this is switched on

# every column has a rule (constraint) so bad data can't get in
# columns that are allowed blanks don't have NOT NULL
# date rules use IS not =, because a fake date gives NULL and NULL would sneak past =
SCHEMA_SQL = """
CREATE TABLE regions (
    region_id       INTEGER PRIMARY KEY,
    region_name     TEXT NOT NULL UNIQUE
);

CREATE TABLE stores (
    store_id        INTEGER PRIMARY KEY,
    store_name      TEXT NOT NULL UNIQUE,
    city            TEXT NOT NULL,
    region_id       INTEGER NOT NULL REFERENCES regions(region_id),
    store_format    TEXT NOT NULL CHECK (store_format IN ('Local', 'High Street', 'Flagship')),
    size_sq_m       REAL NOT NULL CHECK (size_sq_m > 0),
    opened_date     TEXT NOT NULL CHECK (opened_date IS date(opened_date))
);

CREATE TABLE products (
    product_id      INTEGER PRIMARY KEY,
    product_name    TEXT NOT NULL UNIQUE,
    category        TEXT NOT NULL CHECK (category IN ('Skincare', 'Haircare', 'Vitamins', 'Pain Relief',
                                                      'Cold & Flu', 'Oral Care', 'Fragrance', 'Baby')),
    brand           TEXT NOT NULL,
    unit_price      REAL NOT NULL CHECK (unit_price > 0),
    is_own_brand    INTEGER NOT NULL CHECK (is_own_brand IN (0, 1))
);

CREATE TABLE customers (
    customer_id         INTEGER PRIMARY KEY,
    first_name          TEXT NOT NULL,
    last_name           TEXT NOT NULL,
    email               TEXT NOT NULL UNIQUE CHECK (email LIKE '%_@_%._%'),
    phone               TEXT CHECK (length(phone) = 11 AND phone LIKE '07%'),
    age                 INTEGER CHECK (age BETWEEN 16 AND 100),
    gender              TEXT NOT NULL CHECK (gender IN ('Female', 'Male', 'Non-binary', 'Prefer not to say')),
    city                TEXT NOT NULL,
    preferred_store_id  INTEGER NOT NULL REFERENCES stores(store_id),
    loyalty_tier        TEXT NOT NULL CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    signup_date         TEXT NOT NULL CHECK (signup_date IS date(signup_date))
);

CREATE TABLE transactions (
    transaction_id        INTEGER PRIMARY KEY,
    customer_id           INTEGER REFERENCES customers(customer_id),
    store_id              INTEGER NOT NULL REFERENCES stores(store_id),
    transaction_datetime  TEXT NOT NULL CHECK (transaction_datetime IS datetime(transaction_datetime)),
    payment_method        TEXT NOT NULL CHECK (payment_method IN ('Contactless', 'Card', 'Mobile Pay', 'Cash', 'Gift Card')),
    satisfaction_rating   INTEGER CHECK (satisfaction_rating BETWEEN 1 AND 5)
);

CREATE TABLE transaction_items (
    transaction_id      INTEGER NOT NULL REFERENCES transactions(transaction_id),
    product_id          INTEGER NOT NULL REFERENCES products(product_id),
    quantity            INTEGER NOT NULL CHECK (quantity BETWEEN 1 AND 20),
    unit_price_at_sale  REAL NOT NULL CHECK (unit_price_at_sale >= 0),
    PRIMARY KEY (transaction_id, product_id)
);

CREATE TABLE store_inventory (
    store_id            INTEGER NOT NULL REFERENCES stores(store_id),
    product_id          INTEGER NOT NULL REFERENCES products(product_id),
    stock_level         INTEGER NOT NULL CHECK (stock_level >= 0),
    last_restock_date   TEXT NOT NULL CHECK (last_restock_date IS date(last_restock_date)),
    PRIMARY KEY (store_id, product_id)
);

CREATE TABLE daily_weather (
    region_id       INTEGER NOT NULL REFERENCES regions(region_id),
    weather_date    TEXT NOT NULL CHECK (weather_date IS date(weather_date)),
    avg_temp_c      REAL CHECK (avg_temp_c BETWEEN -15 AND 40),
    PRIMARY KEY (region_id, weather_date)
);
"""

conn.executescript(SCHEMA_SQL)

# list the tables to check they were all made
tables = conn.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()
print("Tables created:", [t[0] for t in tables])

Tables created: ['regions', 'stores', 'products', 'customers', 'transactions', 'transaction_items', 'store_inventory', 'daily_weather']


In [218]:
# parent tables go in first, so foreign keys always have something real to point to
TABLES = [
    ('regions', regions_df),
    ('stores', stores_df),
    ('products', products_df),
    ('customers', customers_df),
    ('transactions', transactions_df),
    ('transaction_items', transaction_items_df),
    ('store_inventory', store_inventory_df),
    ('daily_weather', daily_weather_df),
]

for name, df in TABLES:
    # 'append' adds rows into the tables we made, so all our rules stay in place
    df.to_sql(name, conn, if_exists='append', index=False)
conn.commit()   # save everything to the file

# count the rows in each table
for name, _ in TABLES:
    count = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"{name:20} {count:>6} rows")

# check every foreign key points to a real row, an empty list means no problems
print("\nForeign key problems:", conn.execute("PRAGMA foreign_key_check").fetchall())

regions                   5 rows
stores                   15 rows
products                250 rows
customers              1200 rows
transactions           4970 rows
transaction_items     12333 rows
store_inventory        2938 rows
daily_weather          1825 rows

Foreign key problems: []


In [219]:
def q(sql):
    """run a SQL query and show the result as a table"""
    return pd.read_sql_query(sql, conn)

# 1. how many blanks are in each column we added missing data to
print("1. Missing values")
print(q("""
    SELECT
        (SELECT COUNT(*) FROM customers WHERE phone IS NULL)                  AS missing_phone,
        (SELECT COUNT(*) FROM customers WHERE age IS NULL)                    AS missing_age,
        (SELECT COUNT(*) FROM transactions WHERE customer_id IS NULL)         AS guest_sales,
        (SELECT COUNT(*) FROM transactions WHERE satisfaction_rating IS NULL) AS missing_rating,
        (SELECT COUNT(*) FROM daily_weather WHERE avg_temp_c IS NULL)         AS missing_temp
"""), "\n")

# 2. find duplicate customers: same phone and surname but different accounts
print("2. Possible duplicate customers")
print(q("""
    SELECT phone, last_name, COUNT(*) AS accounts, GROUP_CONCAT(customer_id) AS customer_ids
    FROM customers
    WHERE phone IS NOT NULL
    GROUP BY phone, last_name
    HAVING COUNT(*) > 1
    LIMIT 5
"""), "\n")

# 3. find duplicate sales: same customer, store and exact time
print("3. Possible duplicate transactions")
print(q("""
    SELECT customer_id, store_id, transaction_datetime, GROUP_CONCAT(transaction_id) AS transaction_ids
    FROM transactions
    GROUP BY customer_id, store_id, transaction_datetime
    HAVING COUNT(*) > 1
    LIMIT 5
"""), "\n")

# 4. join three tables: money taken each month next to the average temperature
print("4. Monthly revenue and temperature")
print(q("""
    WITH sales AS (
        SELECT strftime('%m', t.transaction_datetime) AS month,
               ROUND(SUM(ti.quantity * ti.unit_price_at_sale), 2) AS revenue
        FROM transactions t
        JOIN transaction_items ti ON t.transaction_id = ti.transaction_id
        GROUP BY month
    ),
    weather AS (
        SELECT strftime('%m', weather_date) AS month, ROUND(AVG(avg_temp_c), 1) AS avg_temp_c
        FROM daily_weather
        GROUP BY month
    )
    SELECT * FROM sales JOIN weather USING (month)
"""), "\n")

# 5. products most often bought together, the starting point for association rules
print("5. Products most often bought together")
print(q("""
    SELECT p1.product_name AS product_a, p2.product_name AS product_b, COUNT(*) AS times_together
    FROM transaction_items a
    JOIN transaction_items b ON a.transaction_id = b.transaction_id AND a.product_id < b.product_id
    JOIN products p1 ON a.product_id = p1.product_id
    JOIN products p2 ON b.product_id = p2.product_id
    GROUP BY a.product_id, b.product_id
    ORDER BY times_together DESC
    LIMIT 5
"""), "\n")

# 6. test the rules work: try to add bad data and check it gets blocked
print("6. Constraint tests")
bad_rows = [
    ("negative price", "INSERT INTO products VALUES (999, 'Test Item', 'Skincare', 'Test', -5.00, 0)"),
    ("rating of 7",    "INSERT INTO transactions VALUES (99999, NULL, 1, '2025-06-01 12:00:00', 'Card', 7)"),
    ("store that doesn't exist", "INSERT INTO transactions VALUES (99999, NULL, 99, '2025-06-01 12:00:00', 'Card', 4)"),
    ("fake date",      "INSERT INTO stores VALUES (99, 'Test Store', 'Test', 1, 'Local', 100, '31/13/2025')"),
]
for label, sql in bad_rows:
    try:
        conn.execute(sql)
        print(f"  {label}: ALLOWED (problem!)")
    except sqlite3.IntegrityError as e:
        print(f"  {label}: blocked ({e})")
conn.rollback()   # undo anything from the tests, just to be safe

1. Missing values
   missing_phone  missing_age  guest_sales  missing_rating  missing_temp
0             72           49          746            3480            18 

2. Possible duplicate customers
         phone last_name  accounts customer_ids
0  07102655449      Cook         2     292,1195
1  07120100290    Hudson         2     670,1192
2  07139428930     Moore         2     756,1198
3  07143650990     Kirby         2     679,1194
4  07147276808      Wood         2     785,1185 

3. Possible duplicate transactions
   customer_id  store_id transaction_datetime transaction_ids
0          NaN         3  2025-11-15 11:32:31       4083,4970
1          NaN         7  2025-06-08 18:56:24       1945,4969
2          NaN        13  2025-04-04 17:53:02       1140,4968
3          NaN        15  2025-12-04 17:51:16       4421,4962
4         83.0        14  2025-01-15 18:33:57        195,4964 

4. Monthly revenue and temperature
   month   revenue  avg_temp_c
0     01  18979.47         5.1
1     

In [220]:
# close the database so everything is saved properly to the file
conn.close()

print("Database size:", round(os.path.getsize(DB_FILE) / 1024 / 1024, 2), "MB")

# download the .db file to the computer, this is the file to submit
from google.colab import files
files.download(DB_FILE)

Database size: 1.07 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>